# SHAP review of the frozen retail forecasting model
This notebook reproduces SHAP contributions from the saved validation forecast inputs. It does not fit or select a model. Attributions describe associations, not causal effects.


In [ ]:
from pathlib import Path
import json, joblib, sys
project = Path.cwd()
if not (project / "module4").is_dir():
    project = project.parent
sys.path.insert(0, str(project))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from module4.train_models import features
from module4.explain_model import grouped_shap, reconstruct_inputs
from module4.setup_tracking import ROOT
parent = '2380b2054b204fff8f19fec80656d2cf'
folder = ROOT / 'models/module4' / parent
xai = max((folder/'xai').iterdir(), key=lambda p: p.stat().st_mtime)
summary = json.loads((xai/'summary.json').read_text())
print(json.dumps(summary, indent=2))


In [ ]:
data = pd.read_parquet(ROOT/'data/processed/prepared.parquet')
data['date'] = pd.to_datetime(data.date)
saved = pd.read_csv(folder/'selected_validation_predictions.csv', parse_dates=['date'])
inputs = reconstruct_inputs(data, saved)
model = joblib.load(folder/'selected_validation_model.joblib')
values, base, raw = grouped_shap(model, features(inputs))
assert np.allclose(base + values.sum(axis=1), raw, atol=1e-5)
print('Verified: SHAP base + contributions reproduce raw model output.')
importance = pd.DataFrame({'feature':features(inputs).columns, 'mean_absolute_shap':np.abs(values).mean(axis=0)}).sort_values('mean_absolute_shap', ascending=False)
display(importance)


In [ ]:
display(Image(filename=str(xai/'global.png')))
display(Image(filename=str(xai/'local.png')))
display(pd.read_csv(xai/'local.csv'))


The global chart averages absolute contributions over 2,800 validation forecasts from 100 series. One-hot contributions are summed within each original category field before taking absolute values. The local plot shows the ten largest contributions; the additivity check includes all features. SHAP explains raw output, while served forecasts clamp negative values to zero. Correlated lags and the reference distribution affect attribution.
